# Phân loại lỗi bề mặt tấm thép — Train trên Colab, tự push từng bước lên GitHub

Notebook này clone repo từ GitHub, train 5 mô hình **One-vs-Rest**, chọn bằng validation và **tự commit + push kết quả lên GitHub**.
Runtime có chết giữa chừng thì kết quả đã nằm trên GitHub, lần sau chỉ cần `--resume` là train tiếp.

Repo: <https://github.com/Vehicle666/steel-plate-fault-classification>

## Chuẩn bị (làm 1 lần): tạo token

1. GitHub → **Settings** → **Developer settings** → **Personal access tokens** → **Fine-grained tokens** → **Generate new token**.
2. **Repository access**: chọn `steel-plate-fault-classification`.
3. **Permissions → Repository permissions → Contents: Read and write**.
4. Copy token (`github_pat_...`).
5. Trong Colab: bấm biểu tượng **🔑 Secrets** (thanh bên trái) → **Add new secret**
   - Name: `GH_TOKEN`
   - Value: token vừa tạo
   - Bật **Notebook access**.

> Token chỉ dùng để push, không bị ghi vào repo. Nếu chưa có secret, cell bên dưới sẽ báo lỗi rõ ràng.

## 1. Nạp token + clone repo

Nếu runtime bị ngắt, chạy lại **từ cell này** rồi nhảy thẳng xuống mục 3 (Resume).

In [ ]:
import os

try:
    from google.colab import userdata

    token = userdata.get("GH_TOKEN")
except Exception as exc:  # noqa: BLE001
    raise SystemExit(
        "Không đọc được secret GH_TOKEN. Hãy thêm secret tên GH_TOKEN "
        "(Personal Access Token, quyền Contents: Read and write) "
        "và bật Notebook access."
    ) from exc

if not token:
    raise SystemExit("Secret GH_TOKEN đang rỗng.")

os.environ["GH_TOKEN"] = token
print("Đã nạp GH_TOKEN từ Colab Secrets (độ dài:", len(token), "ký tự)")

In [ ]:
import os
import subprocess

REPO = "https://github.com/Vehicle666/steel-plate-fault-classification.git"
PROJECT = REPO.rsplit("/", 1)[-1].removesuffix(".git")


def run(*args):
    print("$", " ".join(args))
    subprocess.run(list(args), check=True)


def out(*args):
    return subprocess.run(list(args), capture_output=True, text=True).stdout.strip()


if os.path.isdir(PROJECT):
    os.chdir(PROJECT)
    run("git", "pull", "--ff-only")
else:
    run("git", "clone", REPO)
    os.chdir(PROJECT)

# Colab không có sẵn user.name/user.email -> lấy từ commit gần nhất
if not out("git", "config", "user.name"):
    subprocess.run(["git", "config", "user.name", out("git", "log", "-1", "--format=%an")])
if not out("git", "config", "user.email"):
    subprocess.run(["git", "config", "user.email", out("git", "log", "-1", "--format=%ae")])

run("git", "log", "--oneline", "-3")
print("Checkpoint:", os.listdir("checkpoints") if os.path.isdir("checkpoints") else "(chưa có)")

In [ ]:
!pip install -q -r requirements.txt
!python -c "import sklearn, pandas, numpy; print('sklearn', sklearn.__version__)"

## 2. Train mới — tự push sau mỗi mô hình

Mỗi ứng viên xong sẽ push `reports/` + `checkpoints/train_state.json`; khi hoàn tất, model tốt nhất và test report cũng được push.
Pipeline chia stratified 60/20/20, chọn theo validation macro-F1, chỉ đánh giá test sau khi chọn model.

Chỉnh tham số được ngay trong lệnh, ví dụ `--test-size 0.2 --validation-size 0.2 --random-state 7`
(lưu ý: đổi cấu hình thì checkpoint cũ không còn dùng được, sẽ train lại từ đầu).

In [ ]:
!python -m src.train --push

## 3. Resume sau khi runtime chết / hết pin / đóng tab

Chạy lại **mục 1** (nạp token + clone) rồi chạy cell dưới. Các mô hình đã xong trong
`checkpoints/train_state.json` sẽ được bỏ qua, chỉ train phần còn thiếu — kể cả
khi `checkpoints/` và `reports/` được kéo về từ GitHub.

In [ ]:
!python -m src.train --resume --push

## 4. Xem kết quả trên GitHub

In [ ]:
import pandas as pd
from IPython.display import Image, display

results = pd.read_csv("reports/results.csv")
display(results)

display(Image("reports/figures/metrics_comparison.png"))
display(Image("reports/figures/confusion_matrices_all.png"))

In [ ]:
from pathlib import Path

for name in ["SVM", "k-NN", "Random Forest", "Extra Trees", "Gradient Boosting"]:
    safe = name.lower().replace(" ", "_").replace("-", "")
    print(f"===== {name} =====")
    print(Path(f"reports/classification_report_{safe}.txt").read_text(encoding="utf-8"))

results = pd.read_csv("reports/results.csv")
best = results.sort_values("validation_f1_macro", ascending=False).iloc[0]["model"]
safe = best.lower().replace(" ", "_").replace("-", "")
print("===== Test report:", best, "=====")
print(Path(f"reports/classification_report_{safe}_test.txt").read_text(encoding="utf-8"))
display(Image(f"reports/figures/confusion_matrix_{safe}_test.png"))